<a href="https://colab.research.google.com/github/diegopaucarv/nlp-ai-tools/blob/main/pipeline_radio_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📻 Pipeline de Análisis de Radio: YAMNet + FunASR + Faster-Whisper

Este notebook está diseñado para procesar archivos largos de audio (ej. 30 minutos). Combina tres herramientas:
1. **YAMNet**: Detecta eventos acústicos (Música, Anuncios/Jingles, Silencios, Efectos) y devuelve marcas de tiempo.
2. **FunASR (SenseVoiceSmall)**: Transcribe el texto incluyendo etiquetas semánticas de audio (ej. `<|BGM|>` para música de fondo).
3. **Faster-Whisper**: Proporciona una transcripción altamente veloz con marcas de tiempo (timestamps) exactas para cada segmento de voz.

**⚠️ IMPORTANTE ANTES DE EMPEZAR:**
Ve al menú superior: `Entorno de ejecución` -> `Cambiar tipo de entorno de ejecución` -> Selecciona **A100**. Esto acelerará el procesamiento de 30 minutos a solo un par de minutos.

In [1]:
# ==========================================
# CELDA 1: INSTALACIÓN DE DEPENDENCIAS (VÍA MODERNA)
# ==========================================
print("⏳ 1/2 Alineando ecosistema matemático (Numpy 2.x / Numba / Pandas)...")
# Forzamos la paz: Numba nuevo para soportar Numpy 2.1+, y el Pandas que Colab exige.
!pip install -q pandas==2.2.2 "numpy>=2.1.0" "numba>=0.61.0"

print("⏳ 2/2 Instalando modelos de Inteligencia Artificial (WhisperX, Demucs, etc)...")
!pip install -q git+https://github.com/m-bain/whisperx.git
!pip install -q demucs librosa transformers torchaudio pyannote.audio
!apt-get install -y ffmpeg

print("✅ Entorno estabilizado. Sigue a la Celda 2.")

⏳ 1/2 Alineando ecosistema matemático (Numpy 2.x / Numba / Pandas)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 23.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
⏳ 2/2 Instalando modelos de Inteligencia Artificial (WhisperX, Demucs, etc)...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
    

In [ ]:
# ==========================================
# CELDA 2: AUTENTICACIÓN Y CARPETAS
# ==========================================

import time
import librosa
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import torch
import gc
import shutil
import subprocess
import whisperx
from transformers import AutoProcessor, AutoModelForAudioClassification
import warnings
warnings.filterwarnings("ignore")

from google.colab import drive, userdata
import os
import subprocess

drive.mount('/content/drive')

print("\n--- AUTENTICACIÓN PARA DIARIZACIÓN ---")
HF_TOKEN = userdata.get('HF_TOKEN')

INPUT_DIR = "/content/drive/MyDrive/audios_radio"
OUTPUT_DIR = "/content/drive/MyDrive/resultados_radio"
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- NUEVA FUNCIÓN PARA CONVERTIR MP4 A WAV ---
def convertir_videos_a_wav(directorio):
    # Agregamos .mpeg y .mpg a la lista de extensiones de video válidas
    extensiones_video = ('.mp4', '.mpeg', '.mpg')
    video_files = [f for f in os.listdir(directorio) if f.lower().endswith(extensiones_video)]

    if video_files:
        print(f"\n🎥 Se detectaron {len(video_files)} archivos de video. Extrayendo audio...")
        for f in video_files:
            video_path = os.path.join(directorio, f)
            wav_path = os.path.join(directorio, os.path.splitext(f)[0] + '.wav')

            # Solo convierte si el .wav no existe previamente
            if not os.path.exists(wav_path):
                print(f"   -> Convirtiendo: {f}")
                # Extrae el audio usando ffmpeg silenciosamente
                subprocess.run([
                    "ffmpeg", "-i", video_path,
                    "-q:a", "0", "-map", "a",
                    wav_path, "-y"
                ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            else:
                print(f"   -> Omitiendo {f} (El audio ya existe)")
# 1. Ejecutar la conversión si hay videos
convertir_videos_a_wav(INPUT_DIR)

# 2. Recolectar todos los audios (incluyendo los nuevos .wav extraídos)
archivos = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.mp3', '.wav', '.m4a'))]

print(f"\n📁 Carpeta de entrada: {INPUT_DIR}")
print(f"📻 Archivos detectados: {len(archivos)}")

# ==========================================
# CELDA 3: MOTOR DE PROCESAMIENTO MULTI-MODELO
# ==========================================

# 🚨 Desactivar GPU para TensorFlow (Reservando VRAM para Whisper, Demucs y VAD)
tf.config.set_visible_devices([], 'GPU')

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def format_srt_time(seconds):
    """Convierte segundos crudos al formato SRT estricto para MAXQDA"""
    hrs, mins = int(seconds // 3600), int((seconds % 3600) // 60)
    secs, msecs = int(seconds % 60), int((seconds % 1) * 1000)
    return f"{hrs:02d}:{mins:02d}:{secs:02d},{msecs:03d}"

# --- 1. SEPARACIÓN DE PISTAS CON SMART BATCHING (HTDemucs) ---
def aislar_voces_temporal(audio_path, chunk_length_sec=1200):
    print("      [+] Aplicando HTDemucs (Aislamiento Vocal)...")

    out_dir = "/tmp/demucs_out"
    os.makedirs(out_dir, exist_ok=True)

    duracion = librosa.get_duration(path=audio_path)
    final_vocals_path = "/tmp/final_stitched_vocals.wav"

    # Si el audio es corto, procesarlo directo
    if duracion <= chunk_length_sec:
        temp_in = "/tmp/temp_mixed.wav"
        shutil.copy(audio_path, temp_in)
        subprocess.run(["demucs", "--two-stems=vocals", "-n", "htdemucs", temp_in, "-o", out_dir],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        vocals_path = os.path.join(out_dir, "htdemucs", "temp_mixed", "vocals.wav")
        shutil.copy(vocals_path, final_vocals_path)

    else:
        print(f"      [!] Audio mayor a {chunk_length_sec//60} min. Aplicando Batching seguro para VRAM...")
        import math
        num_chunks = math.ceil(duracion / chunk_length_sec)
        chunk_vocals_list = []

        for i in range(num_chunks):
            start_time = i * chunk_length_sec
            chunk_in = f"/tmp/chunk_{i}.wav"
            print(f"          -> Procesando Lote {i+1}/{num_chunks} (Demucs)...")

            # Cortar audio con precisión usando ffmpeg
            subprocess.run([
                "ffmpeg", "-y", "-i", audio_path,
                "-ss", str(start_time), "-t", str(chunk_length_sec),
                "-c", "copy", chunk_in
            ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

            # Procesar el chunk con Demucs
            subprocess.run(["demucs", "--two-stems=vocals", "-n", "htdemucs", chunk_in, "-o", out_dir],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

            # Guardar la ruta del resultado
            vocal_chunk_path = os.path.join(out_dir, "htdemucs", f"chunk_{i}", "vocals.wav")
            chunk_vocals_list.append(vocal_chunk_path)
            clear_memory() # Limpiar VRAM entre lotes

        # Unir (Stitch) los chunks vocales usando ffmpeg concat para que los timestamps no se rompan
        concat_file = "/tmp/concat_list.txt"
        with open(concat_file, 'w') as f:
            for cv_path in chunk_vocals_list:
                f.write(f"file '{cv_path}'\n")

        subprocess.run([
            "ffmpeg", "-y", "-f", "concat", "-safe", "0",
            "-i", concat_file, "-c", "copy", final_vocals_path
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    return final_vocals_path, out_dir

# --- 2. YAMNET (Contexto Acústico Original) ---
def procesar_yamnet_srt(audio_path, modelo, class_names):
    wav_data, sr = librosa.load(audio_path, sr=16000, mono=True)
    scores, _, _ = modelo(wav_data)

    bloques = []
    estado_actual = None

    for i, frame_scores in enumerate(scores.numpy()):
        top_class = frame_scores.argmax()
        if frame_scores[top_class] < 0.35: continue

        label = class_names[top_class]
        if "Music" in label: etiqueta = "🎵 [MÚSICA DE FONDO / CORTINILLA]"
        elif "Speech" in label: etiqueta = "🗣️ [LOCUCIÓN]"
        elif "Silence" in label: etiqueta = "🔇 [SILENCIO]"
        elif "Animal" in label or "Bird" in label: etiqueta = "🦜 [SONIDO AMBIENTE / NATURALEZA]"
        else: etiqueta = f"🔊 [EFECTO: {label.upper()}]"

        if etiqueta != estado_actual:
            tiempo_seg = i * 0.48
            if etiqueta != "🗣️ [LOCUCIÓN]":
                bloques.append({
                    'tiempo': tiempo_seg,
                    'fin': tiempo_seg + 2.5,
                    'texto': etiqueta
                })
            estado_actual = etiqueta

    del wav_data, scores; clear_memory()
    return bloques

# --- 3. WHISPERX (Diarización Quirúrgica sobre Voces Limpias) ---
def transcribir_y_diarizar(audio_path, hf_token):
    device = "cuda"
    os.environ["HF_TOKEN"] = hf_token

    audio = whisperx.load_audio(audio_path)

    model = whisperx.load_model("small", device, compute_type="float16", language="es")
    result = model.transcribe(audio, batch_size=32, language="es")
    del model; clear_memory()

    model_a, metadata = whisperx.load_align_model(language_code="es", device=device)
    result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)
    del model_a; clear_memory()

    from whisperx.diarize import DiarizationPipeline, assign_word_speakers

    diarize_model = DiarizationPipeline(device=device)
    diarize_segments = diarize_model(audio)
    result = assign_word_speakers(diarize_segments, result)

    del diarize_model; clear_memory()

    bloques = []
    for s in result["segments"]:
        bloques.append({
            'tiempo': s["start"],
            'fin': s["end"],
            'texto': s["text"].strip(),
            'speaker': s.get("speaker", "SPEAKER_UNKNOWN")
        })
    return bloques

# --- 4. EXTRACCIÓN VAD CALIBRADA POR HABLANTE ---
def extraer_emocion_vad(audio_path, segmentos_diarizados, processor, vad_model):
    wav_completo, sr = librosa.load(audio_path, sr=16000, mono=True)
    window_size, step_size = 3.0, 1.0
    valores_crudos = []

    for seg in segmentos_diarizados:
        start_sample, end_sample = int(seg['tiempo'] * 16000), int(seg['fin'] * 16000)

        # Validación de seguridad: Evitar recortes fuera de los límites del arreglo
        if start_sample >= len(wav_completo): continue
        end_sample = min(end_sample, len(wav_completo))

        chunk = wav_completo[start_sample:end_sample]
        duracion_chunk = len(chunk) / 16000

        if duracion_chunk <= window_size:
            inputs = processor(chunk, sampling_rate=16000, return_tensors="pt").to("cuda")
            with torch.no_grad():
                logits = vad_model(**inputs).logits
            a, d, v = logits[0].tolist()
        else:
            v_list, a_list, d_list = [], [], []
            for start_sec in np.arange(0, duracion_chunk - window_size + 0.1, step_size):
                slide = chunk[int(start_sec*16000):int((start_sec+window_size)*16000)]
                if len(slide) == 0: continue
                inputs = processor(slide, sampling_rate=16000, return_tensors="pt").to("cuda")
                with torch.no_grad():
                    logits = vad_model(**inputs).logits
                a_list.append(logits[0][0].item())
                d_list.append(logits[0][1].item())
                v_list.append(logits[0][2].item())
            a = np.mean(a_list) if a_list else 0
            d = np.mean(d_list) if d_list else 0
            v = np.mean(v_list) if v_list else 0

        valores_crudos.append({'seg': seg, 'speaker': seg['speaker'], 'v': v, 'a': a, 'd': d})

    speaker_stats = {}
    speakers_unicos = set(item['speaker'] for item in valores_crudos)

    for spk in speakers_unicos:
        spk_items = [i for i in valores_crudos if i['speaker'] == spk]
        speaker_stats[spk] = {
            'v_m': np.mean([i['v'] for i in spk_items]), 'v_s': np.std([i['v'] for i in spk_items]) or 0.01,
            'a_m': np.mean([i['a'] for i in spk_items]), 'a_s': np.std([i['a'] for i in spk_items]) or 0.01,
            'd_m': np.mean([i['d'] for i in spk_items]), 'd_s': np.std([i['d'] for i in spk_items]) or 0.01,
        }

    def clasificar(val, m, s):
        if val > m + (0.5 * s): return "ALTA"
        elif val < m - (0.5 * s): return "BAJA"
        else: return "MEDIA"

    segmentos_enriquecidos = []
    for item in valores_crudos:
        st = speaker_stats[item['speaker']]
        tag_emocion = f"[VALENCIA: {clasificar(item['v'], st['v_m'], st['v_s'])} | ACTIVACIÓN: {clasificar(item['a'], st['a_m'], st['a_s'])} | DOMINANCIA: {clasificar(item['d'], st['d_m'], st['d_s'])}]"
        texto_dialogo = f"{item['speaker']}: {item['seg']['texto']}"

        segmentos_enriquecidos.append({
            'tiempo': item['seg']['tiempo'],
            'fin': item['seg']['fin'],
            'texto': f"{tag_emocion}\n{texto_dialogo}"
        })

    del wav_completo; clear_memory()
    return segmentos_enriquecidos

# ==========================================
# CELDA 4: EJECUCIÓN DEL PIPELINE Y EXPORTACIÓN SRT
# ==========================================
if not archivos:
    print("⚠️ Sube audios a la carpeta de entrada primero.")
else:
    # --- MODELOS ---
    print("\n⏳ Cargando modelos iniciales en memoria...")
    yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')
    class_names = [line.strip().split(',')[2] for line in tf.io.gfile.GFile(yamnet_model.class_map_path().numpy().decode('utf-8')) if not line.startswith('index')]

    model_id = "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim"
    vad_processor = AutoProcessor.from_pretrained(model_id)
    vad_model = AutoModelForAudioClassification.from_pretrained(model_id).to("cuda")

    for archivo in archivos:
        print(f"\n🎙️ PROCESANDO: {archivo}")
        ruta_audio_mixto = os.path.join(INPUT_DIR, archivo)

        # 1. YAMNet (Audio original)
        print("   [1/4] Mapeando eventos acústicos (YAMNet)...")
        bloques_yamnet = procesar_yamnet_srt(ruta_audio_mixto, yamnet_model, class_names)

        # 2. HTDemucs (Lavar el audio)
        print("   [2/4] Eliminando ruido y música (HTDemucs)...")
        ruta_voces, temp_in, out_dir = aislar_voces_temporal(ruta_audio_mixto)

        # 3. WhisperX (Diarización sobre voces limpias)
        print("   [3/4] Transcribiendo y separando locutores (WhisperX)...")
        bloques_whisper = transcribir_y_diarizar(ruta_voces, HF_TOKEN)

        # 4. VAD (Emoción sobre voces limpias)
        print("   [4/4] Calibrando emociones intrapersonales (Wav2Vec2)...")
        bloques_finales_voz = extraer_emocion_vad(ruta_voces, bloques_whisper, vad_processor, vad_model)

        # LIMPIEZA DE ARCHIVOS TEMPORALES DEMUCS
        shutil.rmtree(out_dir)
        os.remove(temp_in)

        # --- GENERACIÓN DE ARCHIVO SRT PARA MAXQDA ---
        print("   [+] Generando archivo .SRT para MAXQDA...")
        ruta_srt = os.path.join(OUTPUT_DIR, f"{os.path.splitext(archivo)[0]}.srt")

        # Fusionar y ordenar por tiempo
        todos_los_bloques = bloques_yamnet + bloques_finales_voz
        todos_los_bloques.sort(key=lambda x: x['tiempo'])

        with open(ruta_srt, 'w', encoding='utf-8') as f:
            for indice, bloque in enumerate(todos_los_bloques, start=1):
                inicio_srt = format_srt_time(bloque['tiempo'])
                fin_srt = format_srt_time(bloque['fin'])
                f.write(f"{indice}\n")
                f.write(f"{inicio_srt} --> {fin_srt}\n")
                f.write(f"{bloque['texto']}\n\n")

        print(f"✅ ¡Completado! Exportado: {ruta_srt}")

    print("\n🎉 PROCESO TOTAL FINALIZADO CON ÉXITO.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- AUTENTICACIÓN PARA DIARIZACIÓN ---

🎥 Se detectaron 1 archivos de video. Extrayendo audio...
   -> Omitiendo GMT20260904-000057_Recording_1366x768 (1).mp4 (El audio ya existe)

📁 Carpeta de entrada: /content/drive/MyDrive/audios_radio
📻 Archivos detectados: 1

⏳ Cargando modelos iniciales en memoria...


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.34k [00:00<?, ?B/s]